In [1]:
import math
import os
import pandas as pd
import polars as pl
import pytorch_lightning as pl
import torch
import yaml
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import CSVLogger, TensorBoardLogger
from torch import nn
from torchgeodemo import autoencoder_train_latent

#note book shows that alexs and stefs code matches when the same seed is used, and cov loss is turned off in Stef's code.


random_seed_number = 20210321  # 2021 UK Census date used as random seed
def set_random_seeds(seed):
    # """Set random seeds for reproducibility."""
    torch.manual_seed(seed)  # Set seed for PyTorch
    # np.random.seed(seed)  # Set seed for NumPy
    # random.seed(seed)  # Set seed for Python's random module

    # # If using a GPU, set seeds for CUDA as well
    # torch.cuda.manual_seed(seed)
    # torch.cuda.manual_seed_all(seed)  # For multi-GPU setups

    # # Ensure deterministic behavior on GPU
    # torch.backends.cudnn.deterministic = True  # Make sure CUDA uses deterministic algorithms
    # torch.backends.cudnn.benchmark = False  # Avoid using CUDA's autotuning for performance (slower, but more reproducible)

# Set the seed
set_random_seeds(seed=random_seed_number)

# save the data
data_table = pd.read_csv("../data/uk_census_all.csv", index_col=0)

epochs = 10

In [2]:
# Set the seed
set_random_seeds(seed=random_seed_number)

bottleneck_sizes = [32]

# Define the base YAML structure as a dictionary
BASE_YAML = {
    "data": {
        "source": "../data/uk_census_all.csv",
        "nickname": "census_geodemo",
        "id_col": "OA"
    },
    "working_dir": "comparison",
    "autoencoder": {
        "nickname": "bottleneck_PLACEHOLDER",
        "version": "0_3",
        "save_latent": "csv",
        "max_epochs": epochs,
        "batch_size": 0.01,
        "use_covariance_loss": False,
        "encoder": {
            "sizes": [256, 128,96],
            "activation": "ReLU",
            "sparse": {
                "topk_k": "PLACEHOLDER",
                "sparsity_loss_weight": 0.01,
            }
        },
        "decoder": {
            "sizes": [96, 128,256],
            "activation": "LeakyReLU"
        }
    }
}

# Output directory for generated YAMLs
OUTPUT_DIR = "../comparison/yamls"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Generate YAML files with varying latent space sizes
for _latent_size in bottleneck_sizes:
    yaml_config = BASE_YAML.copy()
    yaml_config["autoencoder"]["nickname"] = f"bottleneck_{_latent_size}"
    yaml_config["autoencoder"]["encoder"]["sparse"]["topk_k"] = _latent_size

    config_path = os.path.join(OUTPUT_DIR, f"config_{_latent_size}.yaml")
    with open(config_path, "w") as yaml_file:
        yaml.dump(yaml_config, yaml_file, default_flow_style=False)

print(f"Generated YAML configurations in {OUTPUT_DIR}")


for _latent_size in bottleneck_sizes:
    yaml_path = f"../comparison/yamls/config_{_latent_size}.yaml"
    autoencoder_train_latent.main(yaml_path, create_latent=False,save_reco_error=False)

Generated YAML configurations in ../comparison/yamls

geodemo_config={'autoencoder': {'batch_size': 0.01, 'decoder': {'activation': 'LeakyReLU', 'sizes': [96, 128, 256]}, 'encoder': {'activation': 'ReLU', 'sizes': [256, 128, 96], 'sparse': {'sparsity_loss_weight': 0.01, 'topk_k': 32}}, 'max_epochs': 10, 'nickname': 'bottleneck_32', 'save_latent': 'csv', 'use_covariance_loss': False, 'version': '0_3'}, 'data': {'id_col': 'OA', 'nickname': 'census_geodemo', 'source': '../data/uk_census_all.csv'}, 'working_dir': 'comparison'}


ae_args={'verbose': True, 'encoder_activation': 'ReLU', 'encoder_sparse': True, 'encoder_sparse_topk_k': 32, 'use_covariance_loss': False, 'decoder_sizes': [96, 128, 256, 190], 'decoder_activation': 'LeakyReLU'}

AutoEncoder(
  (dcc_criterion): MSELoss()
  (encoder): MLP(
    (mlp): Sequential(
      (0): Linear(in_features=190, out_features=256, bias=True)
      (1): LeakyReLU(negative_slope=0.01)
      (2): Linear(in_features=256, out_features=128, bias=True)
   

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX A500 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type    | Params | Mode 
--------------------------------------------------
0 | dcc_criterion | MSELoss | 0      | train
1 | encoder       | MLP     | 94.2 K | train
2 | decoder       | MLP     | 94.3 K | train
--------------------------------------------------
188 K     Trainable params
0         Non-trainable params
188 K     Total params
0.754     Total estimated model params size (MB)
18        Modules in train mode
0         Modules in eval mode

Epoch 9: 100%|██████████| 100/100 [00:02<00:00, 47.00it/s, v_num=_125, recon_loss_step=0.0128, sparsity_loss_step=0.135, train_loss_step=0.0141, recon_loss_epoch=0.0134, sparsity_loss_epoch=0.137, train_loss_epoch=0.0148]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 100/100 [00:02<00:00, 46.66it/s, v_num=_125, recon_loss_step=0.0128, sparsity_loss_step=0.135, train_loss_step=0.0141, recon_loss_epoch=0.0134, sparsity_loss_epoch=0.137, train_loss_epoch=0.0148]


In [3]:
# Set the seed
set_random_seeds(seed=random_seed_number)
# Number of Columns
#sc = subset_df.shape[1]
sc = data_table.shape[1]

# ----------------------------------------------------------------------

# TopK activation function
# by Gao et al (2024)
# https://arxiv.org/abs/2406.04093
#
# Based on
# https://github.com/openai/sparse_autoencoder
# MIT license

class TopK(nn.Module):
    def __init__(self, k: int) -> None:
        super().__init__()
        self.k = k
        self.postact_fn = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        topk = torch.topk(x, k=self.k, dim=-1)
        values = self.postact_fn(topk.values)
        # make all other values 0
        result = torch.zeros_like(x)
        result.scatter_(-1, topk.indices, values)
        return result

def normalized_mean_squared_error(
    reconstruction: torch.Tensor,
    original_input: torch.Tensor,
) -> torch.Tensor:
    """
    :param reconstruction: output of Autoencoder.decode (shape: [batch, n_inputs])
    :param original_input: input of Autoencoder.encode (shape: [batch, n_inputs])
    :return: normalized mean squared error (shape: [1])
    """
    return (
        ((reconstruction - original_input) ** 2).mean(dim=1) / (original_input**2).mean(dim=1)
    ).mean()

def normalized_L1_loss(
    latent_activations: torch.Tensor,
    original_input: torch.Tensor,
) -> torch.Tensor:
    """
    :param latent_activations: output of Autoencoder.encode (shape: [batch, n_latents])
    :param original_input: input of Autoencoder.encode (shape: [batch, n_inputs])
    :return: normalized L1 loss (shape: [1])
    """
    return (latent_activations.abs().sum(dim=1) / original_input.norm(dim=1)).mean()

# ----------------------------------------------------------------------
def train_ae(df_scaled):
  
    # Autoencoder as LightningModule
    class Autoencoder(pl.LightningModule):
        def __init__(self, input_size):
            super(Autoencoder, self).__init__()
            # Encoder
            self.encoder = nn.Sequential(
                nn.Linear(input_size, 256),
                nn.LeakyReLU(),
                nn.Linear(256, 128),
                nn.LeakyReLU(),
                nn.Linear(128, 96),
                TopK(32)
            )
            # Decoder
            self.decoder = nn.Sequential(
                nn.Linear(96, 128),
                nn.LeakyReLU(),
                nn.Linear(128, 256),
                nn.LeakyReLU(),
                nn.Linear(256, input_size),
            )
            # Sparsity
            self.sparsity_loss_weight = 0.01
        
        def forward(self, input):
            embeddings = self.encoder(input)
            reconstruction = self.decoder(embeddings)
            return embeddings, reconstruction

        def training_step(self, batch, batch_idx):
            embeddings, reconstruction = self.forward(batch)
            # print(embeddings.shape)
            # print(reconstruction.shape)
            # print(embeddings)
            # print(reconstruction)
            # #stop the code here
            # raise RuntimeError("Stopping execution")
            #error
            # Reconstruction loss
            loss = normalized_mean_squared_error(reconstruction, batch)
            self.log('recon_loss', loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)     


            sparsity_loss = normalized_L1_loss(embeddings, batch)
            self.log('sparsity_loss', sparsity_loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
            loss += self.sparsity_loss_weight * sparsity_loss
            self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
            # Return loss
            return loss

        def configure_optimizers(self):
            optimizer = torch.optim.AdamW(self.parameters(), lr=1e-3)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.2, patience=10, min_lr=1e-5)
            return {'optimizer': optimizer, 'lr_scheduler': scheduler, 'monitor': 'train_loss_epoch'}


    #data_tensor = torch.tensor(subset_df.values).float()
    data_tensor = torch.tensor(df_scaled.values).float()
    data_tensor_nrow = data_tensor.shape[0]
    data_tensor_loader = torch.utils.data.DataLoader(
        data_tensor, 
        batch_size=math.ceil(data_tensor_nrow*0.01), 
        shuffle=True
        )


    logger_test_name = "RAE"
    logger_folder = "lightning_logs"
    logger_tb = TensorBoardLogger(logger_folder, name=logger_test_name)
    logger_csv = CSVLogger(logger_folder, name=logger_test_name)

    autoencoder = Autoencoder(
        input_size = sc
        )
    model_str = str(autoencoder)
    print(model_str, flush=True)

    trainer = Trainer(
        devices=1, 
        accelerator='gpu',
        logger=[logger_tb, logger_csv], 
        max_epochs=epochs
        )

    trainer.fit(
        model=autoencoder, 
        train_dataloaders=data_tensor_loader
        )


    with torch.no_grad():
        autoencoder.eval()
        
        reconstruction_errors = []
        
        for batch in data_tensor_loader:
            batch = batch.to(autoencoder.device)
            reconstructed_batch = autoencoder(batch)

            if isinstance(reconstructed_batch, tuple):
                reconstructed_batch = reconstructed_batch[1] 
            error = torch.mean((batch - reconstructed_batch) ** 2, dim=1)  # MSE per sample
            reconstruction_errors.extend(error)

    reconstruction_errors = torch.stack(reconstruction_errors).cpu().numpy()
    return reconstruction_errors.mean()


train_ae(data_table)

Autoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=190, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=128, out_features=96, bias=True)
    (5): TopK(
      (postact_fn): ReLU()
    )
  )
  (decoder): Sequential(
    (0): Linear(in_features=96, out_features=128, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=128, out_features=256, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=256, out_features=190, bias=True)
  )
)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type       | Params | Mode 
-----------------------------------------------
0 | encoder | Sequential | 94.2 K | train
1 | decoder | Sequential | 94.3 K | train
-----------------------------------------------
188 K     Trainable params
0         Non-trainable params
188 K     Total params
0.754     Total estimated model params size (MB)
14        Modules in train mode
0         Modules in eval mode


Epoch 9: 100%|██████████| 100/100 [00:01<00:00, 54.12it/s, v_num=_281, recon_loss_step=0.0128, sparsity_loss_step=0.135, train_loss_step=0.0141, recon_loss_epoch=0.0134, sparsity_loss_epoch=0.137, train_loss_epoch=0.0148]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 100/100 [00:01<00:00, 53.69it/s, v_num=_281, recon_loss_step=0.0128, sparsity_loss_step=0.135, train_loss_step=0.0141, recon_loss_epoch=0.0134, sparsity_loss_epoch=0.137, train_loss_epoch=0.0148]


np.float32(0.0010660724)

In [ ]:
#this code doesnt agree for some reason

# # Set the seed
# set_random_seeds(seed=random_seed_number)

# # current_seed = torch.initial_seed()
# # print(f"Current seed: {current_seed}")
# # Number of Columns
# #sc = subset_df.shape[1]
# sc = data_table.shape[1]

# # ----------------------------------------------------------------------

# # TopK activation function
# # by Gao et al (2024)
# # https://arxiv.org/abs/2406.04093
# #
# # Based on
# # https://github.com/openai/sparse_autoencoder
# # MIT license

# class TopK(nn.Module):
#     def __init__(self, k: int) -> None:
#         super().__init__()
#         self.k = k
#         self.postact_fn = nn.ReLU()

#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         topk = torch.topk(x, k=self.k, dim=-1)
#         values = self.postact_fn(topk.values)
#         # make all other values 0
#         result = torch.zeros_like(x)
#         result.scatter_(-1, topk.indices, values)
#         return result

# def normalized_mean_squared_error(
#     reconstruction: torch.Tensor,
#     original_input: torch.Tensor,
# ) -> torch.Tensor:
#     """
#     :param reconstruction: output of Autoencoder.decode (shape: [batch, n_inputs])
#     :param original_input: input of Autoencoder.encode (shape: [batch, n_inputs])
#     :return: normalized mean squared error (shape: [1])
#     """
#     return (
#         ((reconstruction - original_input) ** 2).mean(dim=1) / (original_input**2).mean(dim=1)
#     ).mean()

# def normalized_L1_loss(
#     latent_activations: torch.Tensor,
#     original_input: torch.Tensor,
# ) -> torch.Tensor:
#     """
#     :param latent_activations: output of Autoencoder.encode (shape: [batch, n_latents])
#     :param original_input: input of Autoencoder.encode (shape: [batch, n_inputs])
#     :return: normalized L1 loss (shape: [1])
#     """
#     return (latent_activations.abs().sum(dim=1) / original_input.norm(dim=1)).mean()

# # ----------------------------------------------------------------------
# def train_ae(df_scaled):


#     class MLP(pl.LightningModule):
#         def __init__(self, 
#                 size_sequence: list[int], 
#                 final_activation: Literal["Identity", "ReLU", "Tanh", "Sigmoid"] = "Identity",
#                 final_activation_topk: bool = False,
#                 final_activation_topk_k: int = None,
#                 negative_slope=0.01
#                 ) -> None:
#             super(MLP, self).__init__()
#             # Set parameters
#             self.size_sequence = size_sequence
#             self.final_activation = final_activation
#             self.final_activation_topk = final_activation_topk
#             self.final_activation_topk_k = final_activation_topk_k
#             # Create Multi-Layer Perceptron based on size sequence
#             self.mlp = torch.nn.Sequential()
#             # Add all layers but the last
#             for i in range(len(self.size_sequence) - 2):
#                 self.mlp.append(
#                     torch.nn.Linear(self.size_sequence[i], self.size_sequence[i + 1]))
#                 self.mlp.append(
#                     torch.nn.LeakyReLU(negative_slope=negative_slope))
#             # Add final layer
#             self.mlp.append(
#                 torch.nn.Linear(self.size_sequence[-2], self.size_sequence[-1]))
#             # Add final activation
#             # If specified, add TopK activation
#             if self.final_activation_topk:
#                 if self.final_activation_topk_k is None:
#                     # If not specified, use half of the output size
#                     self.final_activation_topk_k = math.floor(self.size_sequence[-1] / 2)
#                 self.mlp.append(
#                     TopK(self.final_activation_topk_k))

#         def forward(self, x: torch.Tensor) -> torch.Tensor:
#             return self.mlp(x)
    

#     # Autoencoder as LightningModule
#     class Autoencoder(pl.LightningModule):
#         def __init__(self, input_size):
#             super(Autoencoder, self).__init__()
#             # Encoder
#             self.encoder = MLP(
#                 size_sequence=[input_size, 256, 128, 96],
#                 final_activation="ReLU",
#                 final_activation_topk=True,
#                 final_activation_topk_k=32
#             )
#             # Decoder
#             self.decoder = MLP(
#                 size_sequence=[96, 128, 256, input_size],
#                 final_activation="LeakyReLU",
#                 final_activation_topk=False
#             )
#             # Sparsity
#             self.sparsity_loss_weight = 0.01
        
#         def forward(self, input):
#             embeddings = self.encoder(input)
#             reconstruction = self.decoder(embeddings)
#             return embeddings, reconstruction

#         def training_step(self, batch, batch_idx):
#             # print("Batch")
#             # print(batch)
#             # print("batch_id: ", batch_idx)
#             embeddings, reconstruction = self.forward(batch)
#             # print("after forward")
#             # print(embeddings.shape)
#             # print(reconstruction.shape)
#             # print(embeddings)
#             # print(reconstruction)
#             # #stop the code here
#             # raise RuntimeError("Stopping execution")
#             #error

#             loss = normalized_mean_squared_error(reconstruction, batch)
#             self.log('recon_loss', loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)     


#             sparsity_loss = normalized_L1_loss(embeddings, batch)
#             self.log('sparsity_loss', sparsity_loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
#             loss += self.sparsity_loss_weight * sparsity_loss
#             self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
#             # Return loss
#             return loss

#         def configure_optimizers(self):
#             optimizer = torch.optim.AdamW(self.parameters(), lr=1e-3)
#             scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.2, patience=10, min_lr=1e-7)
#             return {'optimizer': optimizer, 'lr_scheduler': scheduler, 'monitor': 'train_loss_epoch'}


#     #data_tensor = torch.tensor(subset_df.values).float()
#     data_tensor = torch.tensor(df_scaled.values).float()
#     data_tensor_nrow = data_tensor.shape[0]


#     data_tensor_loader = torch.utils.data.DataLoader(
#         data_tensor, 
#         batch_size=math.ceil(data_tensor_nrow*0.01), 
#         shuffle=True
#         )

#     for batch in data_tensor_loader:
#         print("Batch")
#         print(batch)
#         print(batch.shape)
#         break
#     # Train

#     logger_test_name = "RAE"
#     logger_folder = "lightning_logs"
#     logger_tb = TensorBoardLogger(logger_folder, name=logger_test_name)
#     logger_csv = CSVLogger(logger_folder, name=logger_test_name)

#     autoencoder = Autoencoder(
#         input_size = sc
#         )
#     model_str = str(autoencoder)
#     print(model_str, flush=True)

#     trainer = Trainer(
#         devices=1, 
#         accelerator='gpu',
#         logger=[logger_tb, logger_csv], 
#         max_epochs=epochs
#         )

#     trainer.fit(
#         model=autoencoder, 
#         train_dataloaders=data_tensor_loader
#         )


#     # with torch.no_grad():
#     #     autoencoder.eval()
        
#     #     reconstruction_errors = []
        
#     #     for batch in data_tensor_loader:
#     #         batch = batch.to(autoencoder.device)
#     #         reconstructed_batch = autoencoder(batch)

#     #         if isinstance(reconstructed_batch, tuple):
#     #             reconstructed_batch = reconstructed_batch[1] 
#     #         error = torch.mean((batch - reconstructed_batch) ** 2, dim=1)  # MSE per sample
#     #         reconstruction_errors.extend(error)

#     # reconstruction_errors = torch.stack(reconstruction_errors).cpu().numpy()
#     # return reconstruction_errors.mean()


# train_ae(data_table)